# Assignment 4: N-gram Language Models for Hindi

This notebook loads tokenized sentences from the JSON dataset, splits them into train, development, and test sets, and builds Unigram, Bigram, Trigram, and Quadrigram models.

In [10]:
import json
import random
from collections import defaultdict, Counter

# 1. Load at least 1,000,000 sentences
# We read line by line because the JSON file is very large (26 GB)
sentences = []
file_path = "../assignment1/sentence_tokenized.json"

print("Loading sentences...")
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line in ["[", "]"]: continue
        if line.endswith(","): line = line[:-1]
        if not line: continue
        try:
            sentence = json.loads(line)
            # Tokenize sentence into words (using simple split for Hindi)
            words = sentence.split()
            if len(words) > 0:
                sentences.append(words)
        except json.JSONDecodeError:
            pass
            
        if len(sentences) >= 1000000:
            break

print(f"Total sentences loaded: {len(sentences)}")

Loading sentences...
Total sentences loaded: 1000000


In [11]:
# 2. Split into Train, Dev, and Test sets
# Dev and Test sets have 1000 sentences each, and training data contains the rest.
random.seed(42)
random.shuffle(sentences)

dev_set = sentences[:1000]
test_set = sentences[1000:2000]
train_set = sentences[2000:]

print(f"Training set size: {len(train_set)} sentences")
print(f"Development set size: {len(dev_set)} sentences")
print(f"Test set size: {len(test_set)} sentences")

# Flatten the training sentences to tokens for n-gram modeling
train_tokens = [token for sentence in train_set for token in sentence]
print(f"Total tokens in training set: {len(train_tokens)}")

Training set size: 998000 sentences
Development set size: 1000 sentences
Test set size: 1000 sentences
Total tokens in training set: 17487602


In [12]:
# 3. Define function to build n-gram model
def build_ngram_model(tokens, n=1):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        ngrams.append(ngram)

    counts = Counter(ngrams) # counts frequency of each ngram.
    model = defaultdict(dict)

    for ngram, freq in counts.items(): # maps prefix -> {next_word: frequency}
        prefix = ngram[:-1] if n > 1 else ()
        model[prefix][ngram[-1]] = freq

    # Normalize to probabilities (conditional probability model)
    for prefix in model:
        total_prefix = sum(model[prefix].values())
        for word in model[prefix]:
            model[prefix][word] /= total_prefix

    return model

In [13]:
# 4a. Unigram Model
unigram_model = build_ngram_model(train_tokens, 1)
print("\nUnigram probabilities (top 10):")
print(list(unigram_model[()].items())[:10])


Unigram probabilities (top 10):
[('प्रमुख', 0.00031365077956371605), ('सचिव', 0.00023433744660931785), ('हेल्थ', 4.328781041563046e-05), ('को', 0.019354225925315546), ('न्यायलय', 2.058601287929586e-06), ('ने', 0.014100446705042806), ('कहा', 0.003115921782757865), ('कि', 0.009347765348273593), ('इससे', 0.0007145061970188938), ('निपटने', 5.8441403229556575e-05)]


In [14]:
# 4b. Bigram Model
bigram_model = build_ngram_model(train_tokens, 2)
print("\nBigram probabilities after ('यह',)")
print(list(bigram_model[("यह",)].items())[:10])


Bigram probabilities after ('यह',)
[('कभी', 0.0004580852038479157), ('बदलाव', 0.0007974075770685941), ('अनुच्छेद', 0.00013572894928827133), ('डॉक्टर', 8.483059330516957e-05), ('मेरी', 0.0008652720517127296), ('ट्रेनिंग', 1.6966118661033917e-05), ('भी', 0.0681868308986953), ('स्कूल', 0.00042415296652584787), ('आई', 5.089835598310175e-05), ('प्रतिभागिता', 1.6966118661033917e-05)]


In [ ]:
# 4c. Trigram Model
trigram_model = build_ngram_model(train_tokens, 3)
print("\nTrigram probabilities after ('यह', 'एक')")
print(list(trigram_model[("यह", "एक")].items())[:10])

In [ ]:
# 4d. Quadrigram Model
quadgram_model = build_ngram_model(train_tokens, 4)
print("\nQuadrigram probabilities after ('यह', 'एक', 'बहुत')")
print(list(quadgram_model[("यह", "एक", "बहुत")].items())[:10])